# Aplicación: clasificación de vinos

## 2.1 Importar librerías
- `numpy`, `pandas`: datos numéricos y tablas.
- `matplotlib`: gráficos.
- `load_wine`: dataset incluido en scikit-learn (178 vinos, 13 medidas químicas, 3 cultivares).
- `train_test_split`: separa entrenamiento y prueba.
- `StandardScaler`: estandariza las variables (media 0, desviación 1).
- `Pipeline`: encadena escalado y modelo.
- `KNeighborsClassifier`: clasificador KNN de scikit-learn.
- `GridSearchCV`, `cross_val_score`: búsqueda de hiperparámetros y validación cruzada.
- `metrics`: herramientas de evaluación.
- `joblib`: guardar y cargar el modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.inspection import DecisionBoundaryDisplay
import joblib

## 2.2 Cargar y explorar los datos
- `load_wine(as_frame=True)`: devuelve los datos como DataFrame.
- `X`: 13 características químicas (alcohol, ácido málico, flavonoides, color, prolina, etc.).
- `y`: cultivar del vino (0, 1 o 2).
- `describe()`: estadísticas (media, mínimo, máximo). Observa que las **escalas son muy distintas** entre variables (la prolina llega a cientos, otras no superan 5), lo que justifica escalar.

In [ ]:
datos = load_wine(as_frame=True)
X = datos.data
y = datos.target
nombres_clases = ['Cultivar 1', 'Cultivar 2', 'Cultivar 3']

print("Forma de X:", X.shape)
print("\nVinos por cultivar:")
print(y.value_counts().sort_index())
X.describe().T[['mean', 'min', 'max']]

## 2.3 Separar entrenamiento y prueba
- `test_size=0.25`: 25% de los datos se reserva para evaluar.
- `stratify=y`: mantiene la proporción de cultivares en ambos conjuntos.
- `random_state=42`: resultados reproducibles.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

## 2.4 La importancia de escalar los datos
- `KNeighborsClassifier(n_neighbors=5)`: KNN con 5 vecinos.
- Primero se entrena **sin escalar** y luego **con `StandardScaler`** dentro de un `Pipeline`.
- `fit()` en KNN solo memoriza los datos; `score()` calcula la precisión.
- La diferencia muestra por qué KNN exige escalado.

In [ ]:
knn_sin_escalar = KNeighborsClassifier(n_neighbors=5)
knn_sin_escalar.fit(X_train, y_train)

knn_escalado = Pipeline([('escalador', StandardScaler()),
                         ('knn', KNeighborsClassifier(n_neighbors=5))])
knn_escalado.fit(X_train, y_train)

print(f"Sin escalar: {knn_sin_escalar.score(X_test, y_test)*100:.2f}%")
print(f"Escalado:    {knn_escalado.score(X_test, y_test)*100:.2f}%")

## 2.5 Elegir el mejor K
- Se prueban valores de K de 1 a 30.
- `cross_val_score(..., cv=5)`: validación cruzada de 5 pliegues; da una estimación más estable que una sola división.
- Se grafica la precisión media para cada K y se elige el valor con mayor precisión.
- En la gráfica se ve el equilibrio entre K pequeño (sobreajuste) y K grande (subajuste).

In [ ]:
valores_k = range(1, 31)
precision_k = []
for k in valores_k:
    pipe = Pipeline([('escalador', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=k))])
    precision_k.append(cross_val_score(pipe, X_train, y_train, cv=5).mean())

mejor_k = valores_k[int(np.argmax(precision_k))]

plt.figure(figsize=(8, 4))
plt.plot(valores_k, precision_k, marker='o')
plt.axvline(mejor_k, color='red', linestyle='--', label=f'Mejor K = {mejor_k}')
plt.xlabel('K (número de vecinos)')
plt.ylabel('Precisión (validación cruzada)')
plt.title('Efecto de K en la precisión')
plt.legend()
plt.show()
print(f"Mejor K: {mejor_k} con {max(precision_k)*100:.2f}%")

## 2.6 Comparar distancias y tipos de voto
- `metric`: euclidean, manhattan o chebyshev (distancia usada para buscar vecinos).
- `weights`: `uniform` (todos los vecinos pesan igual) o `distance` (los más cercanos pesan más).
- Se prueban todas las combinaciones con validación cruzada y se muestran en una tabla.

In [ ]:
filas = []
for metrica in ['euclidean', 'manhattan', 'chebyshev']:
    for pesos in ['uniform', 'distance']:
        pipe = Pipeline([('escalador', StandardScaler()),
                         ('knn', KNeighborsClassifier(n_neighbors=mejor_k,
                                                      metric=metrica, weights=pesos))])
        acc = cross_val_score(pipe, X_train, y_train, cv=5).mean()
        filas.append({'Distancia': metrica, 'Pesos': pesos, 'Precisión CV (%)': round(acc*100, 2)})

pd.DataFrame(filas).sort_values('Precisión CV (%)', ascending=False)

## 2.7 Optimizar con GridSearchCV
- `param_grid`: valores a probar. El prefijo `knn__` indica que el parámetro es del paso `knn` del pipeline.
- `GridSearchCV`: evalúa todas las combinaciones con validación cruzada y conserva la mejor.
- `best_params_`: mejores hiperparámetros; `best_estimator_`: modelo ya entrenado con ellos.

In [ ]:
pipe = Pipeline([('escalador', StandardScaler()),
                 ('knn', KNeighborsClassifier())])

param_grid = {
    'knn__n_neighbors': list(range(1, 26)),
    'knn__weights': ['uniform', 'distance'],
    'knn__metric': ['euclidean', 'manhattan']
}

busqueda = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
busqueda.fit(X_train, y_train)

print("Mejores parámetros:", busqueda.best_params_)
print(f"Mejor precisión en validación cruzada: {busqueda.best_score_*100:.2f}%")
mejor_modelo = busqueda.best_estimator_

## 2.8 Evaluar con los datos de prueba
- `predict()`: calcula, para cada vino de prueba, sus K vecinos y vota la clase.
- `classification_report`: precisión, exhaustividad (recall) y F1 por clase.
- `confusion_matrix`: muestra qué cultivares se confunden entre sí.

In [ ]:
pred = mejor_modelo.predict(X_test)

print(f"Precisión en prueba: {accuracy_score(y_test, pred)*100:.2f}%\n")
print(classification_report(y_test, pred, target_names=nombres_clases, digits=3))

ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=nombres_clases).plot(cmap='Blues', colorbar=False)
plt.title('Matriz de confusión')
plt.show()

## 2.9 Visualizar la frontera de decisión según K
- Para dibujar en 2D se usan solo `alcohol` y `flavanoids`.
- Se entrena KNN con K = 1, 5, 15 y 50.
- `DecisionBoundaryDisplay.from_estimator`: colorea la región que el modelo asigna a cada clase.
- Con **K = 1** la frontera es irregular y se ajusta a cada punto (sobreajuste); con **K = 50** es suave pero pierde detalle (subajuste).

In [ ]:
cols = ['alcohol', 'flavanoids']
X2_train, X2_test = X_train[cols].values, X_test[cols].values

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, k in zip(axes, [1, 5, 15, 50]):
    clf = Pipeline([('escalador', StandardScaler()),
                    ('knn', KNeighborsClassifier(n_neighbors=k))])
    clf.fit(X2_train, y_train)
    DecisionBoundaryDisplay.from_estimator(clf, X2_train, response_method='predict',
                                           cmap='viridis', alpha=0.3, ax=ax,
                                           xlabel='Alcohol', ylabel='Flavonoides')
    ax.scatter(X2_train[:, 0], X2_train[:, 1], c=y_train, cmap='viridis',
               edgecolors='k', s=18)
    ax.set_title(f"K = {k} | Prueba: {clf.score(X2_test, y_test)*100:.1f}%")
plt.tight_layout()
plt.show()

## 2.10 Convertir el modelo en una aplicación
Se guarda el modelo y se crea una función que, además de predecir, **explica la decisión mostrando los vecinos**:
1. `joblib.dump` / `joblib.load`: guardan y cargan el modelo (incluido el escalador).
2. `predict_proba`: porcentaje de vecinos de cada clase (confianza de la predicción).
3. `modelo['escalador'].transform`: aplica al vino nuevo el mismo escalado del entrenamiento.
4. `modelo['knn'].kneighbors`: devuelve las distancias y posiciones de los vecinos más cercanos.
5. Con esas posiciones se consulta `y_train` para ver el cultivar de cada vecino.

In [ ]:
joblib.dump(mejor_modelo, 'modelo_knn_vinos.joblib')
modelo_cargado = joblib.load('modelo_knn_vinos.joblib')

def predecir_vino(modelo, medidas, y_referencia):
    """Recibe las 13 medidas de un vino y devuelve su cultivar,
    mostrando la confianza y los vecinos que justifican la decisión."""
    fila = pd.DataFrame([medidas])[list(X.columns)]
    clase = modelo.predict(fila)[0]
    proba = modelo.predict_proba(fila)[0]

    fila_esc = modelo['escalador'].transform(fila)
    distancias, posiciones = modelo['knn'].kneighbors(fila_esc)
    vecinos = y_referencia.iloc[posiciones[0]].values

    print(f"Cultivar predicho: {nombres_clases[clase]}")
    print("Confianza:", {nombres_clases[i]: f"{p*100:.0f}%" for i, p in enumerate(proba)})
    print("Cultivar de los vecinos más cercanos:", [nombres_clases[v] for v in vecinos])
    print("Distancias:", np.round(distancias[0], 2))
    return clase

# Ejemplo con tres vinos del conjunto de prueba
for i in range(3):
    print(f"--- Vino {i+1} (real: {nombres_clases[y_test.iloc[i]]}) ---")
    predecir_vino(modelo_cargado, X_test.iloc[i], y_train)
    print()